# Extract Unity Catalog Data Contracts

Extracts the live Unity Catalog table inventory into compact JSON contracts used by public documentation generation.

Steps performed:
1. Select layers and output directory.
2. Load `.env`, UC runtime config, and canonical table contract through `tools/py_utils`.
3. Query Unity Catalog for current table metadata in each selected layer schema.
4. Keep only documentation-facing table and column fields.
5. Write `docs/data_contracts/<layer>.json` and regenerate dataflow docs.


In [ ]:
# -----------------------------------------------------------------------------
# Manual setup
# -----------------------------------------------------------------------------
LAYERS = ["bronze", "silver", "gold"]
OUTPUT_DIR = "docs/data_contracts"
VERBOSE = True


In [ ]:
# -----------------------------------------------------------------------------
# Imports and project path setup
# -----------------------------------------------------------------------------
from datetime import datetime, timezone
from pathlib import Path
import json
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find project root from current notebook path.")

from tools.docs import generate_dataflow_docs  # noqa: E402
from tools.py_utils.paths import project_root  # noqa: E402
from tools.py_utils.uc_client import UCClient  # noqa: E402
from tools.py_utils.uc_config import catalog_name, load_uc_bulk_config, uc_client_config  # noqa: E402
from tools.py_utils.uc_contracts import layer_schema_names, load_contract  # noqa: E402


In [ ]:
# -----------------------------------------------------------------------------
# Helper functions for compact documentation contracts
# -----------------------------------------------------------------------------
def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def compact_column(column):
    return {
        "name": column.get("name"),
        "type_name": column.get("type_name"),
        "type_text": column.get("type_text"),
        "position": column.get("position"),
        "nullable": column.get("nullable"),
        "comment": column.get("comment"),
    }


def compact_table(table, layer, catalog):
    columns = sorted(table.get("columns") or [], key=lambda item: int(item.get("position") or 0))
    return {
        "name": table.get("name") or table.get("table_name"),
        "catalog": table.get("catalog_name") or catalog,
        "schema": table.get("schema_name") or layer,
        "table_type": table.get("table_type"),
        "data_source_format": table.get("data_source_format"),
        "storage_location": table.get("storage_location"),
        "comment": table.get("comment"),
        "columns": [compact_column(column) for column in columns],
    }


In [ ]:
# -----------------------------------------------------------------------------
# Extract UC state into docs/data_contracts
# -----------------------------------------------------------------------------
root = project_root(__file__ if "__file__" in globals() else PROJECT_ROOT)
contract = load_contract()
config = load_uc_bulk_config()
catalog = catalog_name(config)
uc = UCClient(uc_client_config(config, verbose=VERBOSE))
output_path = root / OUTPUT_DIR
output_path.mkdir(parents=True, exist_ok=True)
extracted_at_utc = utc_now_iso()
written = []

for layer in LAYERS:
    layer_tables = []
    for schema_name in layer_schema_names(contract, layer):
        layer_tables.extend(uc.list_tables(schema_name))
    payload = {
        "layer": layer,
        "catalog": catalog,
        "schema": layer,
        "source": uc.base_url,
        "extracted_at_utc": extracted_at_utc,
        "tables": sorted(
            [compact_table(table, layer, catalog) for table in layer_tables],
            key=lambda item: item["name"] or "",
        ),
    }
    path = output_path / f"{layer}.json"
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(f"Wrote {path}: tables={len(payload['tables'])}")
    written.append(path)

written


In [ ]:
# -----------------------------------------------------------------------------
# Regenerate repository dataflow documentation from refreshed contracts
# -----------------------------------------------------------------------------
generate_dataflow_docs.main()
